In [ ]:
"""
Select fields from CISA ICS advisories CSV and write a trimmed CSV.

Extracted columns (output names exactly as shown):
- product
- vendor
- critical_infrastructure_sector
- ics-cert_number
- cve_id
- CVSS_severity
- Cumulative_CVSS

Usage:
  python3 extract_cisa_ics_adv.py input.csv output.csv

Notes:
- Column matching is case-insensitive and whitespace/punctuation-insensitive.
- Several common header variants are supported (see CANDIDATES below).
"""

import sys
import re
import pandas as pd
from pathlib import Path

# -------- Helpers --------
def norm(s: str) -> str:
    """Normalize header: lowercase, remove non-alphanum."""
    return re.sub(r'[^a-z0-9]+', '', str(s).strip().lower())

def pick_col(df, candidates):
    cols_by_norm = {norm(c): c for c in df.columns}
    for cand in candidates:
        if norm(cand) in cols_by_norm:
            return cols_by_norm[norm(cand)]
    # Last chance: try contains-style partials (rarely needed; keep safe)
    for c in df.columns:
        if any(norm(cand) in norm(c) for cand in candidates):
            return c
    return None

# Desired outputs and acceptable header variants
CANDIDATES = {
    "product": [
        "product", "product name", "ics product", "products"
    ],
    "vendor": [
        "vendor", "vendor name", "manufacturer", "company"
    ],
    "critical_infrastructure_sector": [
        "critical_infrastructure_sector", "critical infrastructure sector",
        "sector", "infrastructure sector"
    ],
    "ics-cert_number": [
        "ics-cert_number", "ics-cert number", "ics-cert advisory number",
        "ics advisory", "ics-cert", "ics number", "ics-advisory",
        "ics advisory number"
    ],
    "cve_id": [
        "cve_id", "cve", "cve id", "cves", "cve list"
    ],
    "CVSS_severity": [
        "cvss_severity", "cvss severity", "severity", "base severity", "cvss v3 severity"
    ],
    "Cumulative_CVSS": [
        "cumulative_cvss", "cumulative cvss", "cumulative score",
        "cvss cumulative", "cvss score total", "cvss total"
    ],
}

def main(argv=None):
    argv = argv or sys.argv[1:]
    if len(argv) < 2:
        print("Usage: python3 extract_cisa_ics_adv.py <input.csv> <output.csv>")
        sys.exit(1)
    in_path = Path(argv[0])
    out_path = Path(argv[1])

    if not in_path.exists():
        print(f"Input file not found: {in_path.resolve()}")
        sys.exit(1)

    # Try UTF-8 first, fallback to latin-1
    try:
        df = pd.read_csv(in_path, dtype=str, keep_default_na=False, na_values=[])
    except UnicodeDecodeError:
        df = pd.read_csv(in_path, dtype=str, keep_default_na=False, na_values=[], encoding="latin-1")

    selected = {}
    missing = []

    for out_name, variants in CANDIDATES.items():
        col = pick_col(df, variants)
        if col is None:
            missing.append(out_name)
        else:
            selected[out_name] = col

    if missing:
        print("ERROR: Could not find the following required fields in the input CSV (case-insensitive):")
        for m in missing:
            print(f"  - {m} (candidates: {CANDIDATES[m]})")
        print("\nAvailable columns in input:")
        for c in df.columns:
            print(f"  - {c}")
        sys.exit(2)

    # Build output DataFrame with standardized headers
    out_df = pd.DataFrame({ out_name: df[selected[out_name]].astype(str) for out_name in CANDIDATES.keys() })

    # Write CSV
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(f"Wrote {len(out_df):,} rows -> {out_path.resolve()}")

if __name__ == "__main__":
    main()
